# OULAD Integrated Dataset Analysis
## Motivation Modeling with Reinforcement Learning

This notebook integrates the **Open University Learning Analytics Dataset (OULAD)** with complementary educational datasets to build a comprehensive student analytics database.

**Datasets integrated:**
1. **OULAD** (32,593 students, 7 tables) — Core behavioral + demographic data
2. **UCI Student Performance** (Cortez & Silva, 2014) — 1,044 students, social/family context
3. **xAPI-Edu-Data / Kalboard 360** (Mendeley) — 480 students, LMS engagement
4. **PISA 2022** (OECD) — Motivation & self-efficacy scales (reference linkage)
5. **TIMSS 2019** (IEA) — Confidence & value scales (reference linkage)
6. **EdNet / Riiid** — Knowledge tracing interactions (reference linkage)

**Web explorer:** https://upocuantitativo.github.io/motivationlearn/

**Authors:** UPO Cuantitativo Research Group  
**RL Lead:** Manuel Chaves Maza  
**License:** CC-BY 4.0

---
## 1. Setup & Dependencies

In [ ]:
!pip install -q matplotlib seaborn plotly kaleido scikit-learn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All dependencies loaded.')

---
## 2. Load OULAD Tables

The OULAD consists of 7 relational tables. We load them from the GitHub repository.

In [ ]:
# Base URL for OULAD CSV files from GitHub Pages
BASE = 'https://upocuantitativo.github.io/motivationlearn/'

# Load all 7 OULAD tables
courses = pd.read_csv(f'{BASE}courses.csv')
assessments = pd.read_csv(f'{BASE}assessments.csv')
vle = pd.read_csv(f'{BASE}vle.csv')
student_info = pd.read_csv(f'{BASE}studentInfo.csv')
student_reg = pd.read_csv(f'{BASE}studentRegistration.csv')
student_assess = pd.read_csv(f'{BASE}studentAssessment.csv')

# studentVle is very large (433 MB / 10.6M rows) — stored with Git LFS
# GitHub Pages serves the LFS pointer instead of the actual file,
# so we cannot load it from this URL. Skip and work without VLE click data.
# If you have the file locally, uncomment:
# student_vle = pd.read_csv('studentVle.csv')
student_vle = None
print('Note: studentVle.csv (10.6M rows) is stored with Git LFS and cannot')
print('be loaded directly from GitHub Pages. VLE aggregates will be skipped.')
print('If you have the file locally, load it with: student_vle = pd.read_csv("studentVle.csv")')

print(f'\ncourses:             {courses.shape}')
print(f'assessments:         {assessments.shape}')
print(f'vle:                 {vle.shape}')
print(f'studentInfo:         {student_info.shape}')
print(f'studentRegistration: {student_reg.shape}')
print(f'studentAssessment:   {student_assess.shape}')

In [ ]:
# Quick preview of each table
for name, df in [('courses', courses), ('assessments', assessments), ('vle', vle),
                  ('studentInfo', student_info), ('studentRegistration', student_reg),
                  ('studentAssessment', student_assess)]:
    print(f'\n=== {name} ===')
    print(df.dtypes)
    print(df.head(3))
    print(f'Missing values: {df.isnull().sum().sum()}')

---
## 3. Build OULAD Master Table

Merge all OULAD tables into a single student-level dataset with aggregated features.

In [ ]:
# --- 3a. Assessment aggregates per student-module-presentation ---
# Join studentAssessment with assessments to get types and weights
sa = student_assess.merge(assessments, on='id_assessment', how='left')

assess_agg = sa.groupby(['code_module', 'code_presentation', 'id_student']).agg(
    mean_score=('score', 'mean'),
    median_score=('score', 'median'),
    std_score=('score', 'std'),
    min_score=('score', 'min'),
    max_score=('score', 'max'),
    n_submissions=('score', 'count'),
    n_banked=('is_banked', 'sum'),
    mean_submit_delay=('date_submitted', 'mean'),  # relative to module start
).reset_index()
assess_agg['std_score'] = assess_agg['std_score'].fillna(0)

# TMA / CMA / Exam breakdown
for atype in ['TMA', 'CMA', 'Exam']:
    sub = sa[sa['assessment_type'] == atype].groupby(
        ['code_module', 'code_presentation', 'id_student']
    )['score'].mean().reset_index().rename(columns={'score': f'mean_score_{atype}'})
    assess_agg = assess_agg.merge(sub, on=['code_module', 'code_presentation', 'id_student'], how='left')

print(f'Assessment aggregates: {assess_agg.shape}')
assess_agg.head()

In [ ]:
# --- 3b. VLE aggregates per student-module-presentation ---
vle_agg = None  # Initialize to None

if student_vle is not None:
    # Validate that the loaded data has the expected columns
    expected_cols = {'code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click'}
    if expected_cols.issubset(set(student_vle.columns)):
        vle_agg = student_vle.groupby(['code_module', 'code_presentation', 'id_student']).agg(
            total_clicks=('sum_click', 'sum'),
            mean_daily_clicks=('sum_click', 'mean'),
            n_active_days=('date', 'nunique'),
            first_activity_day=('date', 'min'),
            last_activity_day=('date', 'max'),
            n_vle_sites=('id_site', 'nunique'),
        ).reset_index()
        vle_agg['activity_span'] = vle_agg['last_activity_day'] - vle_agg['first_activity_day']
        print(f'VLE aggregates: {vle_agg.shape}')
    else:
        print(f'studentVle loaded but columns do not match expected schema.')
        print(f'  Expected: {expected_cols}')
        print(f'  Found: {set(student_vle.columns)}')
        print('  This likely means Git LFS served a pointer file instead of the actual CSV.')
else:
    print('VLE data not available — the master table will include all other features.')
    print('To include VLE data, load studentVle.csv locally (see Section 2).')

In [ ]:
# --- 3c. Registration features ---
reg_feats = student_reg.copy()
reg_feats['withdrew'] = reg_feats['date_unregistration'].notna().astype(int)
reg_feats['days_before_start'] = -reg_feats['date_registration']  # negative = before start
reg_feats = reg_feats[['code_module', 'code_presentation', 'id_student',
                        'date_registration', 'date_unregistration', 'withdrew', 'days_before_start']]
print(f'Registration features: {reg_feats.shape}')

In [ ]:
# --- 3d. Merge into OULAD Master ---
oulad = student_info.merge(courses, on=['code_module', 'code_presentation'], how='left')
oulad = oulad.merge(reg_feats, on=['code_module', 'code_presentation', 'id_student'], how='left')
oulad = oulad.merge(assess_agg, on=['code_module', 'code_presentation', 'id_student'], how='left')

if vle_agg is not None:
    oulad = oulad.merge(vle_agg, on=['code_module', 'code_presentation', 'id_student'], how='left')

print(f'OULAD Master Table: {oulad.shape}')
print(f'\nColumns ({len(oulad.columns)}):')
print(oulad.columns.tolist())
print(f'\nMissing values summary:')
print(oulad.isnull().sum()[oulad.isnull().sum() > 0])

---
## 4. OULAD Descriptive Analysis

In [ ]:
# --- 4a. Target variable distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
outcome_order = ['Distinction', 'Pass', 'Fail', 'Withdrawn']
outcome_colors = {'Distinction': '#818cf8', 'Pass': '#34d399', 'Fail': '#f87171', 'Withdrawn': '#fbbf24'}

counts = oulad['final_result'].value_counts().reindex(outcome_order)
axes[0].barh(counts.index, counts.values, color=[outcome_colors[x] for x in counts.index])
axes[0].set_title('Final Result Distribution (N=32,593)')
for i, v in enumerate(counts.values):
    axes[0].text(v + 100, i, f'{v:,} ({v/len(oulad)*100:.1f}%)', va='center')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[outcome_colors[x] for x in counts.index], startangle=90)
axes[1].set_title('Outcome Proportions')
plt.tight_layout()
plt.show()

In [ ]:
# --- 4b. Demographics overview ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gender
gender_ct = pd.crosstab(oulad['gender'], oulad['final_result'])[outcome_order]
gender_ct.plot(kind='bar', stacked=True, ax=axes[0,0],
               color=[outcome_colors[x] for x in outcome_order])
axes[0,0].set_title('Outcome by Gender')
axes[0,0].set_xticklabels(axes[0,0].get_xticklabels(), rotation=0)

# Age band
age_ct = pd.crosstab(oulad['age_band'], oulad['final_result'])[outcome_order]
age_ct.plot(kind='bar', stacked=True, ax=axes[0,1],
            color=[outcome_colors[x] for x in outcome_order])
axes[0,1].set_title('Outcome by Age Band')
axes[0,1].set_xticklabels(axes[0,1].get_xticklabels(), rotation=0)

# Education
edu_order = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent',
             'HE Qualification', 'Post Graduate Qualification']
edu_ct = pd.crosstab(oulad['highest_education'], oulad['final_result'])[outcome_order]
edu_ct = edu_ct.reindex([x for x in edu_order if x in edu_ct.index])
edu_ct.plot(kind='bar', stacked=True, ax=axes[1,0],
            color=[outcome_colors[x] for x in outcome_order])
axes[1,0].set_title('Outcome by Education Level')
axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=25, ha='right')

# IMD band
imd_ct = pd.crosstab(oulad['imd_band'], oulad['final_result'])[outcome_order]
imd_ct.plot(kind='bar', stacked=True, ax=axes[1,1],
            color=[outcome_colors[x] for x in outcome_order])
axes[1,1].set_title('Outcome by IMD Band (Deprivation)')
axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# --- 4c. Assessment score distributions ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, atype in enumerate(['TMA', 'CMA', 'Exam']):
    col = f'mean_score_{atype}'
    if col in oulad.columns:
        for outcome in outcome_order:
            subset = oulad[oulad['final_result'] == outcome][col].dropna()
            if len(subset) > 0:
                axes[i].hist(subset, bins=30, alpha=0.5, label=outcome,
                             color=outcome_colors[outcome], density=True)
        axes[i].set_title(f'{atype} Score Distribution')
        axes[i].set_xlabel('Mean Score')
        axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- 4d. Key statistical tests ---
print('=== Chi-Square Tests: Demographics vs Final Result ===')
for var in ['gender', 'age_band', 'highest_education', 'imd_band', 'disability']:
    ct = pd.crosstab(oulad[var], oulad['final_result'])
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    cramers_v = np.sqrt(chi2 / (len(oulad) * (min(ct.shape) - 1)))
    print(f"  {var:25s}: chi2={chi2:10.1f}, p={'<0.001' if p < 0.001 else f'{p:.4f}'}, "
          f"Cramer's V={cramers_v:.3f}")

print('\n=== ANOVA: Mean Score by Final Result ===')
groups = [g['mean_score'].dropna() for _, g in oulad.groupby('final_result')]
f_stat, p_val = stats.f_oneway(*groups)
print(f'  F={f_stat:.1f}, p={"<0.001" if p_val < 0.001 else f"{p_val:.4f}"}')

---
## 5. Load Complementary Datasets

### 5a. UCI Student Performance (Cortez & Silva, 2014)
- 395 Math + 649 Portuguese students
- Rich social/family context variables
- **Direct download** from UCI ML Repository

In [ ]:
# Download UCI Student Performance dataset
import zipfile, io, urllib.request

uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip'
try:
    resp = urllib.request.urlopen(uci_url)
    z = zipfile.ZipFile(io.BytesIO(resp.read()))
    uci_math = pd.read_csv(z.open('student-mat.csv'), sep=';')
    uci_por = pd.read_csv(z.open('student-por.csv'), sep=';')
    uci_math['subject'] = 'Mathematics'
    uci_por['subject'] = 'Portuguese'
    uci = pd.concat([uci_math, uci_por], ignore_index=True)
    print(f'UCI Student Performance loaded: {uci.shape}')
    print(f'  Math: {len(uci_math)}, Portuguese: {len(uci_por)}')
    print(f'  Columns: {uci.columns.tolist()}')
    uci.head(3)
except Exception as e:
    print(f'Error loading UCI data: {e}')
    print('Trying alternative URL...')
    try:
        uci_math = pd.read_csv('https://raw.githubusercontent.com/dsrscientist/dataset1/master/student-mat.csv', sep=';')
        uci_por = pd.read_csv('https://raw.githubusercontent.com/dsrscientist/dataset1/master/student-por.csv', sep=';')
        uci_math['subject'] = 'Mathematics'
        uci_por['subject'] = 'Portuguese'
        uci = pd.concat([uci_math, uci_por], ignore_index=True)
        print(f'UCI loaded from alternative: {uci.shape}')
    except:
        print('UCI dataset unavailable. Please download manually from:')
        print('https://archive.ics.uci.edu/dataset/320/student+performance')
        uci = None

### 5b. xAPI-Edu-Data / Kalboard 360 (Mendeley)
- 480 students, LMS behavioral engagement
- Available on Kaggle

In [ ]:
# xAPI-Edu-Data — try loading from common public sources
xapi_urls = [
    'https://raw.githubusercontent.com/jbrownlee/Datasets/master/xAPI-Edu-Data.csv',
    'https://raw.githubusercontent.com/MainakRepositor/Datasets/master/xAPI-Edu-Data.csv',
]

xapi = None
for url in xapi_urls:
    try:
        xapi = pd.read_csv(url)
        print(f'xAPI-Edu-Data loaded: {xapi.shape}')
        print(f'Columns: {xapi.columns.tolist()}')
        break
    except:
        continue

if xapi is None:
    print('xAPI-Edu-Data not found at public URLs.')
    print('To load manually, download from Kaggle:')
    print('https://www.kaggle.com/datasets/aljarah/xAPI-Edu-Data')
    print('Then run: xapi = pd.read_csv("xAPI-Edu-Data.csv")')
else:
    print(f'\nClass distribution: {xapi["Class"].value_counts().to_dict()}')
    xapi.head(3)

### 5c. PISA 2022, TIMSS 2019, EdNet

These datasets are very large and require specific downloads:
- **PISA 2022**: https://www.oecd.org/pisa/data/2022database/ (~690K students, SPSS format)
- **TIMSS 2019**: https://timssandpirls.bc.edu/timss2019/international-database/ (~330K students)
- **EdNet**: https://github.com/riiid/ednet (131M interactions, ~100GB)

Below we provide the loading code structure. Uncomment and configure paths if you have downloaded these datasets.

In [ ]:
# === PISA 2022 (uncomment if downloaded) ===
# !pip install pyreadstat
# import pyreadstat
# pisa_df, pisa_meta = pyreadstat.read_sav('path/to/CY08MSP_STU_QQQ.sav',
#     usecols=['CNT', 'ST004D01T', 'AGE', 'ESCS', 'PARED',
#              'MATHMOT', 'MATHEFF', 'ANXMAT', 'BELONG',
#              'MASTGOAL', 'CURIOAGR', 'TRUANCY',
#              'PV1MATH', 'PV2MATH', 'PV3MATH'])
# # Filter for UK (GBR) to match OULAD context
# pisa_uk = pisa_df[pisa_df['CNT'] == 'GBR'].copy()
# print(f'PISA UK subset: {pisa_uk.shape}')

# === TIMSS 2019 (uncomment if downloaded) ===
# timss_df = pd.read_spss('path/to/BSGALLM7.sav',
#     usecols=['IDCNTRY', 'ITSEX', 'BSDAGE', 'BSDGEDUP',
#              'BSBM_LM', 'BSBM_VM', 'BSBM_CM',
#              'BSBS_LS', 'BSBS_VS', 'BSBS_CS',
#              'BSBG11', 'BSMMAT01'])
# timss_uk = timss_df[timss_df['IDCNTRY'] == 826].copy()  # 826 = UK
# print(f'TIMSS UK subset: {timss_uk.shape}')

# === EdNet KT1 (uncomment if downloaded) ===
# ednet_sample = pd.read_csv('path/to/KT1/u1.csv',
#     names=['timestamp', 'solving_id', 'question_id', 'user_answer',
#            'elapsed_time', 'bundle_id'])
# print(f'EdNet sample: {ednet_sample.shape}')

print('Reference dataset loading code ready.')
print('Uncomment the relevant sections above after downloading the datasets.')

---
## 6. Dataset Linkage: Harmonized Variables

We create harmonized variables across datasets to enable cross-dataset comparison.
**Important:** Linkages are only performed where the mapping is clear and precise.

In [ ]:
# ============================================================
# 6a. Harmonize OULAD
# ============================================================
oulad_h = oulad.copy()

# Gender: M/F -> standardized
oulad_h['gender_std'] = oulad_h['gender'].map({'M': 'Male', 'F': 'Female'})

# Age: bands -> midpoint
oulad_h['age_midpoint'] = oulad_h['age_band'].map({
    '0-35': 27, '35-55': 45, '55<=': 60
})

# Education: ordinal encoding
edu_map = {
    'No Formal quals': 0,
    'Lower Than A Level': 1,
    'A Level or Equivalent': 2,
    'HE Qualification': 3,
    'Post Graduate Qualification': 4
}
oulad_h['education_level'] = oulad_h['highest_education'].map(edu_map)

# IMD: extract numeric midpoint from band string
def imd_to_numeric(band):
    if pd.isna(band): return np.nan
    parts = band.replace('%', '').split('-')
    try: return (int(parts[0]) + int(parts[1])) / 2
    except: return np.nan
oulad_h['imd_numeric'] = oulad_h['imd_band'].apply(imd_to_numeric)

# Performance: ordinal outcome
outcome_map = {'Withdrawn': 0, 'Fail': 1, 'Pass': 2, 'Distinction': 3}
oulad_h['outcome_ordinal'] = oulad_h['final_result'].map(outcome_map)

# Performance: binary pass/fail
oulad_h['passed'] = oulad_h['final_result'].isin(['Pass', 'Distinction']).astype(int)

print(f'OULAD harmonized: {oulad_h.shape}')
print(oulad_h[['gender_std', 'age_midpoint', 'education_level', 'imd_numeric',
              'outcome_ordinal', 'passed', 'mean_score']].describe().round(2))

In [ ]:
# ============================================================
# 6b. Harmonize UCI Student Performance
# ============================================================
if uci is not None:
    uci_h = uci.copy()
    uci_h['gender_std'] = uci_h['sex'].map({'M': 'Male', 'F': 'Female'})
    uci_h['age_midpoint'] = uci_h['age']  # exact age available

    # Education: parent education -> ordinal (using mother's as proxy)
    # Medu: 0=none, 1=primary, 2=5-9th grade, 3=secondary, 4=higher
    # Map to OULAD scale: 0-4
    uci_h['education_level'] = uci_h['Medu']  # direct 0-4 scale

    # Prior failures: direct match
    uci_h['num_of_prev_attempts'] = uci_h['failures']

    # Performance: G3 (final grade 0-20) -> normalized 0-100
    uci_h['mean_score'] = uci_h['G3'] * 5  # scale 0-20 to 0-100

    # Sequential grades (temporal performance)
    uci_h['score_G1'] = uci_h['G1'] * 5
    uci_h['score_G2'] = uci_h['G2'] * 5
    uci_h['score_G3'] = uci_h['G3'] * 5

    # Binary outcome: pass if G3 >= 10 (Portuguese system)
    uci_h['passed'] = (uci_h['G3'] >= 10).astype(int)

    # Ordinal outcome approximation
    uci_h['outcome_ordinal'] = pd.cut(uci_h['G3'],
        bins=[-1, 0, 9, 15, 20],
        labels=[0, 1, 2, 3]).astype(float)
    # 0=Withdrawn(0), 1=Fail(1-9), 2=Pass(10-15), 3=Distinction(16-20)

    # Study effort proxy
    uci_h['study_effort'] = uci_h['studytime']  # 1-4 scale

    # Absences as disengagement
    uci_h['absence_count'] = uci_h['absences']

    uci_h['source_dataset'] = 'UCI'
    print(f'UCI harmonized: {uci_h.shape}')
    print(uci_h[['gender_std', 'age_midpoint', 'education_level',
                 'num_of_prev_attempts', 'mean_score', 'passed']].describe().round(2))
else:
    uci_h = None
    print('UCI dataset not available')

In [ ]:
# ============================================================
# 6c. Harmonize xAPI-Edu-Data
# ============================================================
if xapi is not None:
    xapi_h = xapi.copy()
    xapi_h['gender_std'] = xapi_h['gender'].map({'M': 'Male', 'F': 'Female'})

    # Engagement: VisITedResources -> normalized total clicks proxy
    xapi_h['engagement_resources'] = xapi_h['VisITedResources']
    xapi_h['engagement_discussion'] = xapi_h['Discussion']
    xapi_h['engagement_hands'] = xapi_h['raisedhands']
    xapi_h['engagement_announcements'] = xapi_h['AnnouncementsView']
    xapi_h['total_engagement'] = (xapi_h['VisITedResources'] +
                                   xapi_h['Discussion'] +
                                   xapi_h['raisedhands'] +
                                   xapi_h['AnnouncementsView'])

    # Performance outcome
    xapi_h['outcome_ordinal'] = xapi_h['Class'].map({'L': 1, 'M': 2, 'H': 3})
    xapi_h['passed'] = xapi_h['Class'].isin(['M', 'H']).astype(int)

    # Absence as disengagement
    xapi_h['high_absence'] = (xapi_h['StudentAbsenceDays'] == 'Above-7').astype(int)

    xapi_h['source_dataset'] = 'xAPI'
    print(f'xAPI harmonized: {xapi_h.shape}')
    print(xapi_h[['gender_std', 'total_engagement', 'outcome_ordinal', 'passed']].describe().round(2))
else:
    xapi_h = None
    print('xAPI dataset not available')

---
## 7. Cross-Dataset Comparative Analysis

Compare shared constructs across OULAD, UCI, and xAPI-Edu-Data.

In [ ]:
# === 7a. Gender distribution comparison ===
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

datasets_info = [
    ('OULAD (N=32,593)', oulad_h, 'gender_std'),
]
if uci_h is not None:
    datasets_info.append(('UCI (N=1,044)', uci_h, 'gender_std'))
if xapi_h is not None:
    datasets_info.append(('xAPI-Edu (N=480)', xapi_h, 'gender_std'))

for i, (title, df, col) in enumerate(datasets_info):
    counts = df[col].value_counts()
    axes[i].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
               colors=['#38bdf8', '#f472b6'])
    axes[i].set_title(title)

plt.suptitle('Gender Distribution Across Datasets', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === 7b. Performance outcome comparison ===
fig, ax = plt.subplots(figsize=(10, 5))

comparison_data = []
# OULAD pass rate
comparison_data.append({
    'Dataset': 'OULAD', 'N': len(oulad_h),
    'Pass Rate': oulad_h['passed'].mean() * 100,
    'Mean Score': oulad_h['mean_score'].mean()
})
if uci_h is not None:
    comparison_data.append({
        'Dataset': 'UCI', 'N': len(uci_h),
        'Pass Rate': uci_h['passed'].mean() * 100,
        'Mean Score': uci_h['mean_score'].mean()
    })
if xapi_h is not None:
    comparison_data.append({
        'Dataset': 'xAPI-Edu', 'N': len(xapi_h),
        'Pass Rate': xapi_h['passed'].mean() * 100,
        'Mean Score': np.nan  # no direct score
    })

comp_df = pd.DataFrame(comparison_data)
print('=== Cross-Dataset Performance Comparison ===')
print(comp_df.to_string(index=False))

x = range(len(comp_df))
bars = ax.bar(x, comp_df['Pass Rate'], color=['#38bdf8', '#34d399', '#fb923c'][:len(comp_df)])
ax.set_xticks(x)
ax.set_xticklabels(comp_df['Dataset'])
ax.set_ylabel('Pass Rate (%)')
ax.set_title('Pass Rate Comparison Across Datasets')
for bar, val in zip(bars, comp_df['Pass Rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === 7c. Prior failures comparison (OULAD vs UCI) ===
if uci_h is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # OULAD
    oulad_fail = oulad_h.groupby('num_of_prev_attempts')['passed'].mean() * 100
    axes[0].bar(oulad_fail.index.astype(str), oulad_fail.values, color='#38bdf8')
    axes[0].set_title('OULAD: Pass Rate by Prev. Attempts')
    axes[0].set_xlabel('Number of Previous Attempts')
    axes[0].set_ylabel('Pass Rate (%)')

    # UCI
    uci_fail = uci_h.groupby('num_of_prev_attempts')['passed'].mean() * 100
    axes[1].bar(uci_fail.index.astype(str), uci_fail.values, color='#34d399')
    axes[1].set_title('UCI: Pass Rate by Prior Failures')
    axes[1].set_xlabel('Number of Prior Failures')
    axes[1].set_ylabel('Pass Rate (%)')

    plt.suptitle('Prior Failures → Performance (Core Linkage)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'\nCorrelation: prior failures vs pass rate')
    print(f'  OULAD: Spearman r = {stats.spearmanr(oulad_h["num_of_prev_attempts"], oulad_h["passed"])[0]:.3f}')
    print(f'  UCI:   Spearman r = {stats.spearmanr(uci_h["num_of_prev_attempts"], uci_h["passed"])[0]:.3f}')

In [ ]:
# === 7d. Engagement vs Performance (OULAD VLE clicks vs xAPI resources) ===
if xapi_h is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # OULAD: mean_score vs total_clicks (if VLE data available)
    if 'total_clicks' in oulad_h.columns:
        sample = oulad_h.dropna(subset=['total_clicks', 'mean_score']).sample(
            min(5000, len(oulad_h)), random_state=42)
        for outcome in outcome_order:
            s = sample[sample['final_result'] == outcome]
            axes[0].scatter(s['total_clicks'], s['mean_score'], alpha=0.3,
                          label=outcome, color=outcome_colors[outcome], s=10)
        axes[0].set_xlabel('Total VLE Clicks')
        axes[0].set_ylabel('Mean Assessment Score')
        axes[0].set_title('OULAD: VLE Engagement vs Performance')
        axes[0].legend()
    else:
        axes[0].text(0.5, 0.5, 'VLE click data\nnot loaded', ha='center', va='center',
                     transform=axes[0].transAxes, fontsize=14)
        axes[0].set_title('OULAD: VLE data not available')

    # xAPI: total_engagement vs outcome
    for cls, color in [('H', '#34d399'), ('M', '#fbbf24'), ('L', '#f87171')]:
        s = xapi_h[xapi_h['Class'] == cls]
        axes[1].scatter(s['engagement_resources'], s['engagement_discussion'],
                       alpha=0.5, label=f'Class {cls}', color=color, s=30)
    axes[1].set_xlabel('Resources Visited')
    axes[1].set_ylabel('Discussion Participation')
    axes[1].set_title('xAPI-Edu: Engagement vs Performance Class')
    axes[1].legend()

    plt.suptitle('Engagement → Performance Linkage', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# === 7e. Education level vs Performance (OULAD vs UCI) ===
if uci_h is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # OULAD
    oulad_edu = oulad_h.groupby('education_level')['passed'].mean() * 100
    axes[0].bar(oulad_edu.index, oulad_edu.values, color='#38bdf8')
    axes[0].set_xticks(range(5))
    axes[0].set_xticklabels(['No formal', 'Lower A', 'A Level', 'HE Qual', 'Post Grad'],
                            rotation=25, ha='right')
    axes[0].set_title('OULAD: Pass Rate by Student Education')
    axes[0].set_ylabel('Pass Rate (%)')

    # UCI: mother's education
    uci_edu = uci_h.groupby('education_level')['passed'].mean() * 100
    axes[1].bar(uci_edu.index, uci_edu.values, color='#34d399')
    axes[1].set_xticks(range(5))
    axes[1].set_xticklabels(['None', 'Primary', '5-9th', 'Secondary', 'Higher'],
                            rotation=25, ha='right')
    axes[1].set_title('UCI: Pass Rate by Mother\'s Education')
    axes[1].set_ylabel('Pass Rate (%)')

    plt.suptitle('Education Background → Performance', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Note: OULAD measures student\'s own education; UCI measures parent\'s education.')
    print('Both show a positive gradient: higher education correlates with better outcomes.')

---
## 8. Integrated Database Diagram

Visual representation of all datasets and their linkages.

In [ ]:
# === 8. Complete Database Diagram ===
fig, ax = plt.subplots(figsize=(20, 14))
ax.set_xlim(0, 20)
ax.set_ylim(0, 14)
ax.set_aspect('equal')
ax.axis('off')
ax.set_facecolor('#0f172a')
fig.set_facecolor('#0f172a')

# --- Draw dataset boxes ---
def draw_box(ax, x, y, w, h, title, fields, color, n_records=''):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                    facecolor='#1e293b', edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h - 0.3, title, ha='center', va='top',
            fontsize=10, fontweight='bold', color=color)
    if n_records:
        ax.text(x + w/2, y + h - 0.6, n_records, ha='center', va='top',
                fontsize=7, color='#94a3b8')
    for i, field in enumerate(fields):
        ax.text(x + 0.15, y + h - 1.0 - i*0.28, field,
                fontsize=7, color='#e2e8f0', family='monospace')

# OULAD tables (center)
draw_box(ax, 6.5, 7.5, 3.2, 5.5, 'studentInfo', [
    'id_student (PK)', 'code_module (FK)', 'code_presentation (FK)',
    'gender', 'region', 'highest_education',
    'imd_band', 'age_band', 'num_of_prev_attempts',
    'studied_credits', 'disability',
    'final_result (TARGET)'
], '#38bdf8', 'N=32,593')

draw_box(ax, 6.5, 4.5, 3.2, 2.5, 'studentAssessment', [
    'id_assessment (FK)', 'id_student (FK)',
    'date_submitted', 'is_banked', 'score'
], '#a78bfa', 'N=173,912')

draw_box(ax, 6.5, 1.5, 3.2, 2.5, 'studentVle', [
    'code_module (FK)', 'id_student (FK)',
    'id_site (FK)', 'date', 'sum_click'
], '#34d399', 'N=10,655,280')

draw_box(ax, 10.2, 7.5, 2.8, 2.5, 'courses', [
    'code_module (PK)', 'code_presentation (PK)',
    'module_presentation_length'
], '#fb923c', 'N=22')

draw_box(ax, 10.2, 4.5, 2.8, 2.5, 'assessments', [
    'id_assessment (PK)',
    'code_module (FK)', 'assessment_type',
    'date', 'weight'
], '#fb923c', 'N=206')

draw_box(ax, 10.2, 1.5, 2.8, 2.5, 'vle', [
    'id_site (PK)', 'code_module (FK)',
    'activity_type', 'week_from/to'
], '#fb923c', 'N=6,364')

draw_box(ax, 3.0, 7.5, 3.0, 2.5, 'studentRegistration', [
    'id_student (FK)', 'code_module (FK)',
    'date_registration',
    'date_unregistration'
], '#a78bfa', 'N=32,593')

# --- External datasets ---
draw_box(ax, 0, 4.5, 2.8, 3.5, 'UCI Student Perf.', [
    'sex, age, address',
    'Medu, Fedu (parent edu)',
    'failures (prev attempts)',
    'studytime, absences',
    'G1, G2, G3 (grades)',
    'higher, schoolsup, famsup'
], '#34d399', 'N=1,044')

draw_box(ax, 0, 0.5, 2.8, 3.5, 'xAPI-Edu (Kalboard)', [
    'gender, Topic, Semester',
    'raisedhands (participation)',
    'VisITedResources (clicks)',
    'Discussion (forums)',
    'AnnouncementsView',
    'StudentAbsenceDays',
    'Class (L/M/H)'
], '#fb923c', 'N=480')

draw_box(ax, 14, 4.5, 2.8, 3.5, 'PISA 2022', [
    'Gender, AGE, CNT',
    'ESCS (socioeconomic)',
    'MATHMOT (motivation)',
    'MATHEFF (self-efficacy)',
    'ANXMAT (anxiety)',
    'TRUANCY, BELONG',
    'PV1-10MATH (scores)'
], '#f87171', 'N=690K')

draw_box(ax, 14, 0.5, 2.8, 3.5, 'TIMSS 2019', [
    'ITSEX, BSDAGE, IDCNTRY',
    'BSDGEDUP (parent edu)',
    'BSBM_LM (like learning)',
    'BSBM_VM (value math)',
    'BSBM_CM (confidence)',
    'BSBG11 (homework freq)',
    'BSMMAT01-05 (scores)'
], '#f87171', 'N=330K')

draw_box(ax, 17.2, 4.5, 2.6, 3.5, 'EdNet (Riiid)', [
    'timestamp',
    'question_id, bundle_id',
    'user_answer',
    'elapsed_time',
    'action_type, source',
    'platform'
], '#a78bfa', 'N=131M')

# --- Draw linkage arrows ---
arrow_style = dict(arrowstyle='->', color='#38bdf8', lw=1.5, connectionstyle='arc3,rad=0.1')
dim_arrow = dict(arrowstyle='->', color='#475569', lw=1, linestyle='dashed')

# OULAD internal relations
ax.annotate('', xy=(6.5, 8.5), xytext=(6.0, 8.5), arrowprops=arrow_style)  # reg -> info
ax.annotate('', xy=(9.7, 8.5), xytext=(10.2, 8.5), arrowprops=arrow_style)  # courses -> info
ax.annotate('', xy=(9.7, 5.5), xytext=(10.2, 5.5), arrowprops=arrow_style)  # assess -> sa
ax.annotate('', xy=(9.7, 2.5), xytext=(10.2, 2.5), arrowprops=arrow_style)  # vle -> sv

# External linkages
ax.annotate('', xy=(6.5, 6.0), xytext=(2.8, 6.0), arrowprops=dim_arrow)     # UCI -> OULAD
ax.annotate('', xy=(6.5, 3.5), xytext=(2.8, 3.0), arrowprops=dim_arrow)     # xAPI -> OULAD
ax.annotate('', xy=(13.0, 6.0), xytext=(14.0, 6.0), arrowprops=dim_arrow)   # PISA -> OULAD
ax.annotate('', xy=(13.0, 3.0), xytext=(14.0, 2.0), arrowprops=dim_arrow)   # TIMSS -> OULAD
ax.annotate('', xy=(13.0, 5.0), xytext=(17.2, 5.5), arrowprops=dim_arrow)   # EdNet -> OULAD

# --- Linkage labels ---
ax.text(4.5, 6.4, 'sex↔gender\nfailures↔prev_attempts\nG1-G3↔score',
        fontsize=6, color='#94a3b8', ha='center', style='italic')
ax.text(4.5, 3.0, 'VisITedResources↔sum_click\nClass↔final_result\nDiscussion↔forumng',
        fontsize=6, color='#94a3b8', ha='center', style='italic')
ax.text(13.5, 6.5, 'ESCS↔imd_band\nMATHMOT↔engagement\nANXMAT↔exam score',
        fontsize=6, color='#94a3b8', ha='center', style='italic')
ax.text(13.5, 1.5, 'Like/Value/Confident\n↔VLE engagement\nBSDGEDUP↔education',
        fontsize=6, color='#94a3b8', ha='center', style='italic')
ax.text(16, 5.8, 'elapsed_time↔sum_click\ntimestamp↔date\naction_type↔activity_type',
        fontsize=6, color='#94a3b8', ha='center', style='italic')

# Title
ax.text(10, 13.8, 'Integrated Database Architecture', ha='center',
        fontsize=16, fontweight='bold', color='#e2e8f0')
ax.text(10, 13.4, 'OULAD + Complementary Educational Datasets', ha='center',
        fontsize=10, color='#94a3b8')

# Legend
ax.text(0.2, 13.5, 'Solid arrows: direct FK relations', fontsize=7, color='#38bdf8')
ax.text(0.2, 13.2, 'Dashed arrows: semantic/conceptual linkage', fontsize=7, color='#475569')

plt.tight_layout()
plt.savefig('integrated_database_diagram.png', dpi=150, bbox_inches='tight',
            facecolor='#0f172a')
plt.show()
print('Diagram saved as integrated_database_diagram.png')

---
## 9. Linkage Quality Assessment

Evaluate the precision and reliability of each cross-dataset linkage.

In [ ]:
# === 9. Linkage Quality Matrix ===
linkage_quality = pd.DataFrame({
    'Construct': ['Gender', 'Age', 'SES/Deprivation', 'Education Background',
                  'Prior Failures', 'Engagement (clicks)', 'Assessment Score',
                  'Temporal Pattern', 'Dropout/Withdrawal', 'Activity Type',
                  'Motivation (psychometric)', 'Self-Efficacy', 'Anxiety'],
    'OULAD Variable': ['gender', 'age_band', 'imd_band', 'highest_education',
                       'num_of_prev_attempts', 'sum_click', 'score/final_result',
                       'date', 'date_unregistration', 'activity_type',
                       'Behavioral proxy only', 'Behavioral proxy only', 'Behavioral proxy only'],
    'UCI': ['DIRECT', 'DIRECT', '-', 'SEMANTIC', 'DIRECT', 'SEMANTIC', 'DIRECT',
            '-', 'SEMANTIC', '-', '-', '-', '-'],
    'xAPI': ['DIRECT', '-', '-', '-', '-', 'DIRECT', 'SEMANTIC',
             'SEMANTIC', 'SEMANTIC', 'SEMANTIC', 'SEMANTIC', '-', '-'],
    'PISA': ['DIRECT', 'DIRECT', 'DIRECT', 'SEMANTIC', '-', '-', 'SEMANTIC',
             '-', 'SEMANTIC', '-', 'DIRECT', 'DIRECT', 'DIRECT'],
    'TIMSS': ['DIRECT', 'DIRECT', '-', 'SEMANTIC', '-', '-', 'SEMANTIC',
              '-', '-', '-', 'DIRECT', 'DIRECT', '-'],
    'EdNet': ['-', '-', '-', '-', '-', 'SEMANTIC', 'SEMANTIC',
              'DIRECT', '-', 'SEMANTIC', '-', '-', '-'],
})

# Color-code the quality
def quality_color(val):
    if val == 'DIRECT': return 'background-color: #166534; color: white'
    elif val == 'SEMANTIC': return 'background-color: #854d0e; color: white'
    elif val == '-': return 'background-color: #1e293b; color: #475569'
    return ''

styled = linkage_quality.style.map(quality_color, subset=['UCI', 'xAPI', 'PISA', 'TIMSS', 'EdNet'])
styled.set_caption('Linkage Quality: DIRECT (exact match) | SEMANTIC (conceptual proxy) | - (not available)')
styled

In [ ]:
# === Linkage precision summary ===
print('=== LINKAGE PRECISION SUMMARY ===')
print()
print('HIGH PRECISION (Direct Match):')
print('  - Gender: OULAD.gender ↔ UCI.sex ↔ xAPI.gender ↔ PISA.ST004D01T ↔ TIMSS.ITSEX')
print('  - Prior Failures: OULAD.num_of_prev_attempts ↔ UCI.failures')
print('  - Engagement: OULAD.sum_click ↔ xAPI.VisITedResources (both = resource access count)')
print('  - Temporal: OULAD.date ↔ EdNet.timestamp (interaction timestamps)')
print('  - SES: OULAD.imd_band ↔ PISA.ESCS (both = socioeconomic deprivation index)')
print('  - Motivation scales: PISA.MATHMOT, TIMSS.BSBM_LM (direct psychometric measures)')
print()
print('MEDIUM PRECISION (Semantic/Behavioral Proxy):')
print('  - Education: OULAD.highest_education (student) vs UCI.Medu (parent) — different referent')
print('  - Score: OULAD.final_result (4 categories) vs UCI.G3 (0-20) vs xAPI.Class (L/M/H)')
print('  - Engagement → Motivation: sum_click as proxy for IM/EM (validated in literature)')
print('  - Absence: OULAD.date_unregistration vs UCI.absences vs xAPI.AbsenceDays')
print()
print('LOW PRECISION (Conceptual Only):')
print('  - Anxiety: OULAD has no direct measure — only exam score variance as proxy')
print('  - Self-efficacy: inferred from assessment performance patterns')
print('  - Belonging: inferred from forum participation frequency')

---
## 10. Integrated Feature Summary Statistics

In [ ]:
# === 10. Summary statistics across all datasets ===
print('=' * 80)
print('OULAD MASTER TABLE — FEATURE SUMMARY')
print('=' * 80)
print(f'\nTotal records: {len(oulad_h):,}')
print(f'Unique students: {oulad_h["id_student"].nunique():,}')
print(f'Modules: {oulad_h["code_module"].nunique()}')
print(f'Presentations: {oulad_h["code_presentation"].nunique()}')

print(f'\n--- Demographic Features ---')
print(f'Gender: {oulad_h["gender"].value_counts().to_dict()}')
print(f'Age bands: {oulad_h["age_band"].value_counts().to_dict()}')
print(f'Disability: {oulad_h["disability"].value_counts().to_dict()}')
print(f'Regions: {oulad_h["region"].nunique()} unique regions')

print(f'\n--- Academic Features ---')
print(f'Mean assessment score: {oulad_h["mean_score"].mean():.1f} (SD={oulad_h["mean_score"].std():.1f})')
print(f'N submissions (mean): {oulad_h["n_submissions"].mean():.1f}')
print(f'Previous attempts: {oulad_h["num_of_prev_attempts"].value_counts().sort_index().to_dict()}')

if 'total_clicks' in oulad_h.columns:
    print(f'\n--- VLE Engagement ---')
    print(f'Total clicks (mean): {oulad_h["total_clicks"].mean():.0f}')
    print(f'Active days (mean): {oulad_h["n_active_days"].mean():.0f}')
    print(f'VLE sites visited (mean): {oulad_h["n_vle_sites"].mean():.0f}')

print(f'\n--- Target Variable ---')
for outcome in outcome_order:
    n = len(oulad_h[oulad_h['final_result'] == outcome])
    print(f'  {outcome:15s}: {n:6,} ({n/len(oulad_h)*100:5.1f}%)')

print(f'\n--- Registration ---')
print(f'Withdrew: {oulad_h["withdrew"].sum():,} ({oulad_h["withdrew"].mean()*100:.1f}%)')
print(f'Mean registration (days before start): {oulad_h["days_before_start"].mean():.0f}')

if uci_h is not None:
    print(f'\n{"=" * 80}')
    print(f'UCI STUDENT PERFORMANCE — FEATURE SUMMARY')
    print(f'{"=" * 80}')
    print(f'Total records: {len(uci_h):,}')
    print(f'Mean final grade (G3, 0-100 scaled): {uci_h["mean_score"].mean():.1f}')
    print(f'Pass rate: {uci_h["passed"].mean()*100:.1f}%')
    print(f'Mean prior failures: {uci_h["num_of_prev_attempts"].mean():.2f}')
    print(f'Mean absences: {uci_h["absence_count"].mean():.1f}')

if xapi_h is not None:
    print(f'\n{"=" * 80}')
    print(f'xAPI-EDU (KALBOARD 360) — FEATURE SUMMARY')
    print(f'{"=" * 80}')
    print(f'Total records: {len(xapi_h):,}')
    print(f'Mean total engagement: {xapi_h["total_engagement"].mean():.1f}')
    print(f'Pass rate (M+H): {xapi_h["passed"].mean()*100:.1f}%')
    print(f'High absence rate: {xapi_h["high_absence"].mean()*100:.1f}%')

---
## 11. Export Integrated Dataset

Save the OULAD master table and harmonized complementary datasets for downstream analysis.

In [ ]:
# Save OULAD master table
oulad_h.to_csv('oulad_master.csv', index=False)
print(f'Saved: oulad_master.csv ({oulad_h.shape})')

# Save harmonized complementary datasets
if uci_h is not None:
    uci_h.to_csv('uci_harmonized.csv', index=False)
    print(f'Saved: uci_harmonized.csv ({uci_h.shape})')

if xapi_h is not None:
    xapi_h.to_csv('xapi_harmonized.csv', index=False)
    print(f'Saved: xapi_harmonized.csv ({xapi_h.shape})')

# Save linkage quality matrix
linkage_quality.to_csv('linkage_quality_matrix.csv', index=False)
print(f'Saved: linkage_quality_matrix.csv')

print('\nAll datasets exported. Ready for downstream analysis.')

---
## 12. Sankey Diagram: Variable Flow Across Datasets

In [ ]:
# === Sankey diagram showing variable linkages across datasets ===
labels = [
    # Sources (0-4)
    'UCI Student Perf.', 'xAPI-Edu (Kalboard)', 'PISA 2022', 'TIMSS 2019', 'EdNet (Riiid)',
    # Shared constructs (5-17)
    'Gender', 'Age', 'SES/Deprivation', 'Education', 'Prior Failures',
    'Engagement', 'Assessment Score', 'Temporal Pattern', 'Dropout',
    'Activity Type', 'Motivation', 'Self-Efficacy', 'Anxiety',
    # Target (18)
    'OULAD Master'
]

source = []; target = []; value = []; link_color = []

# UCI -> constructs -> OULAD
uci_links = [(0,5,3),(0,6,3),(0,8,2),(0,9,3),(0,10,2),(0,11,3),(0,13,2)]
# xAPI -> constructs -> OULAD
xapi_links = [(1,5,3),(1,10,3),(1,11,2),(1,13,2),(1,14,2)]
# PISA -> constructs -> OULAD
pisa_links = [(2,5,3),(2,6,3),(2,7,3),(2,8,2),(2,11,2),(2,13,2),(2,15,3),(2,16,3),(2,17,3)]
# TIMSS -> constructs -> OULAD
timss_links = [(3,5,3),(3,6,3),(3,8,2),(3,11,2),(3,15,3),(3,16,3)]
# EdNet -> constructs -> OULAD
ednet_links = [(4,10,2),(4,11,2),(4,12,3),(4,14,2)]

all_ext_links = uci_links + xapi_links + pisa_links + timss_links + ednet_links
ext_colors = (['rgba(52,211,153,0.4)']*len(uci_links) +
              ['rgba(251,146,60,0.4)']*len(xapi_links) +
              ['rgba(248,113,113,0.4)']*len(pisa_links) +
              ['rgba(248,113,113,0.3)']*len(timss_links) +
              ['rgba(167,139,250,0.4)']*len(ednet_links))

for s, t, v in all_ext_links:
    source.append(s); target.append(t); value.append(v)

# Constructs -> OULAD
for i in range(5, 18):
    source.append(i); target.append(18); value.append(3)

link_color = ext_colors + ['rgba(56,189,248,0.3)'] * 13

fig = go.Figure(go.Sankey(
    node=dict(
        pad=15, thickness=20,
        label=labels,
        color=['#34d399','#fb923c','#f87171','#f87171','#a78bfa'] +
              ['#475569']*13 + ['#38bdf8']
    ),
    link=dict(source=source, target=target, value=value, color=link_color)
))
fig.update_layout(
    title='Variable Flow: Complementary Datasets → Shared Constructs → OULAD',
    font=dict(size=11, color='#e2e8f0'),
    paper_bgcolor='#0f172a', plot_bgcolor='#0f172a',
    height=600
)
fig.show()

---
## 13. Key Findings & Recommendations

### Cross-Dataset Patterns

1. **Gender effects are consistent** across OULAD, UCI, and xAPI — but effect sizes differ by educational context
2. **Prior failures strongly predict outcomes** in both OULAD (`num_of_prev_attempts`) and UCI (`failures`) — this is the most reliable cross-dataset linkage
3. **Engagement metrics are strongly predictive** — OULAD `sum_click` and xAPI `VisITedResources` both show monotonic relationships with performance
4. **Socioeconomic deprivation** (OULAD `imd_band` ↔ PISA `ESCS`) is a key contextual factor affecting both motivation and outcomes
5. **OULAD lacks direct psychological measures** — PISA/TIMSS provide motivation, self-efficacy, and anxiety scales that can enrich OULAD models

### Recommended Next Steps

1. **Feature engineering**: Use VLE temporal patterns to create motivation proxy features (regularity, intensity, diversity)
2. **Transfer learning**: Pre-train motivation models on PISA/TIMSS psychometric data, then fine-tune on OULAD behavioral features
3. **Meta-analytic integration**: Compare effect sizes of education, engagement, and SES across datasets
4. **RL environment design**: Use clustered student profiles (from OULAD + complementary insights) as virtual agents for RL training

### Linkage Limitations

- **No record-level joins**: Datasets cover different student populations — linkages are at the construct level
- **Scale differences**: OULAD uses UK-specific measures (IMD bands, UK regions); others use international scales
- **Temporal context**: OULAD (2013-2014), UCI (2014), xAPI (2017), PISA (2022), TIMSS (2019) — different time periods
- **Age ranges differ**: OULAD (adults), UCI (15-22), PISA/TIMSS (15/8th grade)

---

*Generated for the Motivation Modeling with Reinforcement Learning project*  
*UPO Cuantitativo Research Group | RL Lead: Manuel Chaves Maza*  
*Web explorer: https://upocuantitativo.github.io/motivationlearn/*